In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 2.3.3
NumPy version: 2.5.3


In [2]:
# Locate project and dataset
project_root = Path.cwd()

# Handle execution from either project root or notebooks/
if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "all_training.csv"

print("Project root:", project_root)
print("Dataset path:", data_path)
print("Dataset exists:", data_path.exists())

Project root: c:\Users\bl\Documents\ML-Projects\clinical-ml
Dataset path: c:\Users\bl\Documents\ML-Projects\clinical-ml\data\raw\all_training.csv
Dataset exists: True


In [3]:
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Number of rows:", len(df))
print("Number of columns:", df.shape[1])

df.head()

Dataset shape: (1736, 36)
Number of rows: 1736
Number of columns: 36


,Unnamed: 0,Sex,Age,CA,CK,CREA,ALP,GGT,GLU,AST,...,MO,EO,BA,NET,LYT,MOT,EOT,BAT,Suspect,target
0,A00345_2020-03-25,1.0,82.0,2.09,NaN,1.150,95.0,40.0,78.0,26.0,...,9.5,2.9,0.5,6.40,1.20,0.8,0.3,0.0,1.0,0
1,A00791_2020-03-19,1.0,51.0,1.97,237.0,0.970,54.0,98.0,98.0,74.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1
2,A00741_2020-03-04,1.0,58.0,2.11,NaN,1.000,80.0,147.0,106.0,41.0,...,7.3,0.3,0.1,5.45,0.75,0.5,0.0,0.0,1.0,0
3,A00605_2020-04-15,0.0,82.0,2.27,138.0,0.755,123.5,176.5,106.0,114.0,...,9.5,1.7,0.9,3.60,2.60,0.7,0.1,0.1,0.5,0
4,A00417_2020-02-24,1.0,79.0,2.07,73.0,1.810,62.0,36.5,96.0,28.0,...,10.0,8.5,0.5,0.40,0.50,0.1,0.1,0.0,1.0,0


In [4]:
print("Columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

Columns:
01. Unnamed: 0
02. Sex
03. Age
04. CA
05. CK
06. CREA
07. ALP
08. GGT
09. GLU
10. AST
11. ALT
12. LDH
13. PCR
14. KAL
15. NAT
16. UREA
17. WBC
18. RBC
19. HGB
20. HCT
21. MCV
22. MCH
23. MCHC
24. PLT1
25. NE
26. LY
27. MO
28. EO
29. BA
30. NET
31. LYT
32. MOT
33. EOT
34. BAT
35. Suspect
36. target


In [10]:
id_col = "Unnamed: 0"

ids = df[id_col].astype(str).str.strip()

def assign_cohort(patient_id):
    if patient_id.startswith("A"):
        return "training"

    elif patient_id.startswith("PSMAY"):
        return "training"

    # Internal-external validation cohort:
    # IDs 2001–2058
    elif patient_id.isdigit() and 2001 <= int(patient_id) <= 2058:
        return "internal_external"

    # Remaining numeric IDs belong to external cohort
    elif patient_id.isdigit():
        return "external"

    else:
        return "unclassified"

df["cohort"] = ids.apply(assign_cohort)

cohort_counts = (
    df["cohort"]
    .value_counts()
    .rename_axis("cohort")
    .reset_index(name="n")
)

cohort_counts

,cohort,n
0,training,1624
1,internal_external,58
2,external,54


In [12]:
expected_counts = {
    "training": 1624,
    "internal_external": 58,
    "external": 54,
}

actual_counts = df["cohort"].value_counts().to_dict()

print("--- COHORT RECONSTRUCTION ---")

for cohort, expected in expected_counts.items():
    actual = actual_counts.get(cohort, 0)
    print(f"{cohort:20s}: {actual:4d} | expected = {expected}")

assert "unclassified" not in actual_counts, "Some records remain unclassified."
assert actual_counts == expected_counts, "Cohort counts do not match expected values."

print("\n✓ Cohort reconstruction successful.")

--- COHORT RECONSTRUCTION ---
training            : 1624 | expected = 1624
internal_external   :   58 | expected = 58
external            :   54 | expected = 54

✓ Cohort reconstruction successful.


In [13]:
outcome_table = pd.crosstab(
    df["cohort"],
    df["target"],
    margins=True
)

outcome_table

target,0,1,All
cohort,,,
external,54,0,54
internal_external,28,30,58
training,838,786,1624
All,920,816,1736


In [14]:
# Remove variables that must not be predictors
non_predictors = [id_col, "target", "cohort"]

predictors = [
    col for col in df.columns
    if col not in non_predictors
]

print("Number of candidate predictors:", len(predictors))
print("\nCandidate predictors:")

for i, col in enumerate(predictors, start=1):
    print(f"{i:02d}. {col}")

Number of candidate predictors: 34

Candidate predictors:
01. Sex
02. Age
03. CA
04. CK
05. CREA
06. ALP
07. GGT
08. GLU
09. AST
10. ALT
11. LDH
12. PCR
13. KAL
14. NAT
15. UREA
16. WBC
17. RBC
18. HGB
19. HCT
20. MCV
21. MCH
22. MCHC
23. PLT1
24. NE
25. LY
26. MO
27. EO
28. BA
29. NET
30. LYT
31. MOT
32. EOT
33. BAT
34. Suspect


In [15]:
# Summarize predictor data types and missingness

feature_summary = pd.DataFrame({
    "dtype": df[predictors].dtypes.astype(str),
    "non_missing": df[predictors].notna().sum(),
    "missing": df[predictors].isna().sum(),
    "missing_pct": df[predictors].isna().mean() * 100,
    "unique_values": df[predictors].nunique()
})

feature_summary["missing_pct"] = (
    feature_summary["missing_pct"].round(2)
)

feature_summary

,dtype,non_missing,missing,missing_pct,unique_values
Sex,float64,1736,0,0.00,3
Age,float64,1682,54,3.11,101
CA,float64,1643,93,5.36,229
CK,float64,704,1032,59.45,327
CREA,float64,1662,74,4.26,422
ALP,float64,1262,474,27.30,256
GGT,float64,1300,436,25.12,291
GLU,float64,1638,98,5.65,321
AST,float64,1638,98,5.65,280
ALT,float64,1640,96,5.53,263


In [16]:
# Inspect special variables before defining the model

for col in ["PCR", "Sex", "Suspect", "target"]:
    print(f"\n{'='*50}")
    print(f"VARIABLE: {col}")
    print(f"{'='*50}")

    print("Data type:", df[col].dtype)
    print("Non-missing:", df[col].notna().sum())
    print("Missing:", df[col].isna().sum())
    print("Unique values:", df[col].nunique())

    print("\nFirst 20 unique values:")
    print(df[col].dropna().unique()[:20])

    print("\nSummary:")
    print(df[col].describe())


VARIABLE: PCR
Data type: float64
Non-missing: 1639
Missing: 97
Unique values: 1055

First 20 unique values:
[ 61.9  116.5   86.65   1.6  317.5  215.1   29.5  146.9   78.2   56.1
  55.5   64.1   17.15  19.     2.8   56.9   61.3  114.65 157.7  177.7 ]

Summary:
count    1639.000000
mean       67.097614
std        77.796202
min         0.300000
25%         6.275000
50%        38.600000
75%       101.400000
max       442.700000
Name: PCR, dtype: float64

VARIABLE: Sex
Data type: float64
Non-missing: 1736
Missing: 0
Unique values: 3

First 20 unique values:
[1.         0.         0.93333333]

Summary:
count    1736.000000
mean        0.573118
std         0.494731
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: Sex, dtype: float64

VARIABLE: Suspect
Data type: float64
Non-missing: 1736
Missing: 0
Unique values: 3

First 20 unique values:
[1.  0.5 0. ]

Summary:
count    1736.000000
mean        0.763537
std         0.374767
min  

In [17]:
# Examine relationship of Sex and Suspect with target

print("--- SEX × TARGET ---")
print(pd.crosstab(df["Sex"], df["target"], margins=True))

print("\n--- SUSPECT × TARGET ---")
print(pd.crosstab(df["Suspect"], df["target"], margins=True))

--- SEX × TARGET ---
target      0    1   All
Sex                     
0.0       442  299   741
0.933333    0    1     1
1.0       478  516   994
All       920  816  1736

--- SUSPECT × TARGET ---
target     0    1   All
Suspect                
0.0      234   37   271
0.5      202   77   279
1.0      484  702  1186
All      920  816  1736


In [18]:
# Investigate special coding in Sex and Suspect

print("--- UNUSUAL SEX VALUES ---")

display(
    df.loc[
        ~df["Sex"].isin([0, 1]),
        [id_col, "Sex", "Suspect", "target", "cohort"]
    ]
)

print("\n--- SEX VALUES BY COHORT ---")
display(
    pd.crosstab(
        df["cohort"],
        df["Sex"],
        margins=True
    )
)

print("\n--- SUSPECT VALUES BY COHORT ---")
display(
    pd.crosstab(
        df["cohort"],
        df["Suspect"],
        margins=True
    )
)

--- UNUSUAL SEX VALUES ---


,Unnamed: 0,Sex,Suspect,target,cohort
1647,2024,0.933333,1.0,1,internal_external



--- SEX VALUES BY COHORT ---


Sex,0.0,0.933333,1.0,All
cohort,,,,
external,27,0,27,54
internal_external,37,1,20,58
training,677,0,947,1624
All,741,1,994,1736



--- SUSPECT VALUES BY COHORT ---


Suspect,0.0,0.5,1.0,All
cohort,,,,
external,34,0,20,54
internal_external,36,0,22,58
training,201,279,1144,1624
All,271,279,1186,1736


In [19]:
# Investigate the PCR variable by cohort and outcome

pcr_summary = (
    df.groupby(["cohort", "target"])["PCR"]
      .agg(["count", "mean", "median", "std", "min", "max"])
      .round(2)
)

display(pcr_summary)

count   mean  median    std   min     max
cohort            target                                           
external          0          48  40.17   15.55  55.64  0.30  301.20
internal_external 0          28  52.03   25.65  54.35  0.50  181.06
                  1          30  68.91   38.20  63.76  2.17  214.03
training          0         756  46.04   14.50  67.44  0.30  385.10
                  1         777  89.72   65.90  83.13  0.30  442.70

In [20]:
# Examine correlations between PCR and target
# within the development cohort only

training_df = df[df["cohort"] == "training"].copy()

print("Training cohort:", training_df.shape)
print(
    "PCR-target correlation:",
    round(training_df["PCR"].corr(training_df["target"]), 3)
)

print("\nPCR summary by COVID status:")
display(
    training_df.groupby("target")["PCR"]
    .describe()
    .round(2)
)

Training cohort: (1624, 37)
PCR-target correlation: 0.277

PCR summary by COVID status:


,count,mean,std,min,25%,50%,75%,max
target,,,,,,,,
0,756.0,46.04,67.44,0.3,1.9,14.5,61.95,385.1
1,777.0,89.72,83.13,0.3,25.2,65.9,128.80,442.7
